In [0]:
!pip install lightgbm

In [0]:
import os
import re
import gc
import time
import shutil
from pathlib import Path

import numpy as np
import pandas as pd

import pyarrow as pa
import pyarrow.dataset as ds

import lightgbm as lgb
import matplotlib.pyplot as plt
import psutil

In [0]:
# ============================================================
# Input paths
# ============================================================

DATA_PATH = (
    "/Volumes/"
    "ph-com-ai-us-prod-catalog-2553213839599380/"
    "kerendia_oe_journey/"
    "lgbm_feature_selection/"
    "feature_selection_mdb_25k_final_parquet"
)

OUTPUT_DIR = (
    "/Volumes/"
    "ph-com-ai-us-prod-catalog-2553213839599380/"
    "kerendia_oe_journey/"
    "lgbm_feature_selection/v2/"
    "lgbm_shap_dependency_outputs"
)

MODEL_PATH = "lightgbm_nbrx_model_wo_segment_new_2.txt"

FEATURE_MAPPING_PATH = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_oe_journey/kerendia_oe_volume/oe_git_offload/Kerendia_OE_2.0/Feature Selection/feature_alias_mapping_shap_dependency.csv"

booster = lgb.Booster(model_file=MODEL_PATH)

feature_mapping_df = pd.read_csv(FEATURE_MAPPING_PATH)

LOCAL_OUTPUT_DIR = "/tmp/lgbm_shap_dependency_outputs"
Path(LOCAL_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# ============================================================
# Target column
# ============================================================

TARGET_COL = "TOTAL_NBRX_C4"

# ============================================================
# Feature selection for dependence plot
# Option 1: give exact feature name
# Option 2: keep FEATURE_NAME = None and search by terms
# ============================================================

FEATURE_NAME_LIST = [
    "EFFICACY_COUNT_P3",
    "F2F_ATTENDED_R_P3",
    "TOTAL_NBRX_P3","TOTAL_TRX_P3","TOTAL_HCP_NBRX_CLAIMS_PRODUCT_GROUP_KERENDIA_IN_LAST_12_MONTH",
    "DOXIMITY_DOCNEWS_EMAIL_HEADLINE_VIEW_R_P3","PD_TRX_P3"
]


# Number of rows used for the SHAP dependence plot.
# Recommended: 50k first.
# Use None for all rows, but it will be slower.
SHAP_DEPENDENCE_ROWS = None

# Arrow reads this many rows per parquet batch.
ARROW_BATCH_SIZE = 2048

# LightGBM SHAP prediction batch size.
# Keep moderate because pred_contrib returns n_features + 1 values.
PRED_BATCH_SIZE = 512

# CPU threads
NUM_THREADS = psutil.cpu_count(logical=False)

if NUM_THREADS is None:
    NUM_THREADS = 64

NUM_THREADS = min(NUM_THREADS, 64)

SEED = 42

print("DATA_PATH:", DATA_PATH)
print("MODEL_PATH:", MODEL_PATH)
print("FEATURE_MAPPING_PATH:", FEATURE_MAPPING_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("NUM_THREADS:", NUM_THREADS)

In [0]:
# ============================================================
# Helper functions
# ============================================================

def safe_filename(x):
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(x))[:180]


def copy_local_to_output(local_path, output_filename=None):
    local_path = str(local_path)

    if output_filename is None:
        output_filename = os.path.basename(local_path)

    final_output_path = f"{OUTPUT_DIR.rstrip('/')}/{output_filename}"

    try:
        dbutils.fs.cp(
            f"file:{local_path}",
            f"dbfs:{final_output_path}",
            True
        )
    except Exception:
        Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
        shutil.copyfile(local_path, final_output_path)

    print("Saved:")
    print(final_output_path)

    return final_output_path


def get_selected_row_indices(n_rows, sample_rows=None, seed=42):
    if sample_rows is None or sample_rows >= n_rows:
        return np.arange(n_rows, dtype=np.int64)

    rng = np.random.default_rng(seed)

    return np.sort(
        rng.choice(
            n_rows,
            size=sample_rows,
            replace=False
        ).astype(np.int64)
    )

In [0]:
# ============================================================
# Load saved LightGBM model
# ============================================================

booster = lgb.Booster(model_file=MODEL_PATH)

print("Model loaded successfully.")
print("Number of model features:", booster.num_feature())
print("Number of trees:", booster.num_trees())
print("Current iteration:", booster.current_iteration())

In [0]:
# ============================================================
# Load feature mapping
# ============================================================

feature_mapping_df = pd.read_csv(FEATURE_MAPPING_PATH)

required_cols = {"feature_index", "feature_alias", "feature"}
missing_cols = required_cols - set(feature_mapping_df.columns)

if len(missing_cols) > 0:
    raise ValueError(f"Feature mapping file missing columns: {missing_cols}")

feature_mapping_df = (
    feature_mapping_df
    .sort_values("feature_index")
    .reset_index(drop=True)
)

feature_cols = feature_mapping_df["feature"].tolist()
feature_aliases = feature_mapping_df["feature_alias"].tolist()

n_features = len(feature_cols)

print("Features in mapping:", n_features)
print("First 10 features:")
print(feature_cols[:10])

if booster.num_feature() != n_features:
    raise ValueError(
        f"Model expects {booster.num_feature()} features, "
        f"but mapping has {n_features} features."
    )

In [0]:
# ============================================================
# Read PyArrow dataset and validate requested features
# ============================================================

dataset = ds.dataset(DATA_PATH, format="parquet")

all_cols = dataset.schema.names

if TARGET_COL not in all_cols:
    raise ValueError(f"Target column not found in parquet: {TARGET_COL}")

missing_model_features = [c for c in feature_cols if c not in all_cols]

if len(missing_model_features) > 0:
    print("Missing model feature count:", len(missing_model_features))
    print("Sample missing model features:")
    print(missing_model_features[:20])
    raise ValueError("Some model features are not present in parquet data.")

# Remove duplicates from feature list while preserving order
FEATURE_NAME_LIST = list(dict.fromkeys(FEATURE_NAME_LIST))

missing_requested_features = [
    f for f in FEATURE_NAME_LIST
    if f not in feature_cols
]

if len(missing_requested_features) > 0:
    print("These requested features were not found in model feature_cols:")
    print(missing_requested_features)
    raise ValueError("Some requested features are not present in model features.")

FEATURE_INDEX_LIST = [
    feature_cols.index(f)
    for f in FEATURE_NAME_LIST
]

print("Features selected for SHAP dependence:")
for f, idx in zip(FEATURE_NAME_LIST, FEATURE_INDEX_LIST):
    print(f"{idx}: {f}")

In [0]:
# ============================================================
# Compute SHAP dependence data for multiple features
# ============================================================

n_rows_total = dataset.count_rows()

selected_row_indices = get_selected_row_indices(
    n_rows=n_rows_total,
    sample_rows=SHAP_DEPENDENCE_ROWS,
    seed=SEED
)

print("Total parquet rows:", n_rows_total)
print("Rows used for SHAP dependence:", len(selected_row_indices))
print("Number of features for dependence:", len(FEATURE_NAME_LIST))

scan_cols = [TARGET_COL] + feature_cols

cast_schema = pa.schema([
    (col, pa.float32())
    for col in scan_cols
])

scanner = dataset.scanner(
    columns=scan_cols,
    batch_size=ARROW_BATCH_SIZE,
    use_threads=True
)

# Store data separately for each requested feature
dependence_store = {
    feature_name: {
        "feature_value": [],
        "shap_value": [],
        "target_value": []
    }
    for feature_name in FEATURE_NAME_LIST
}

global_row_start = 0
selected_pointer = 0

start_time = time.time()

for batch_id, record_batch in enumerate(scanner.to_batches(), start=1):
    batch_n = record_batch.num_rows
    global_row_end = global_row_start + batch_n

    # Find sampled rows that fall inside this batch
    left = np.searchsorted(selected_row_indices, global_row_start, side="left")
    right = np.searchsorted(selected_row_indices, global_row_end, side="left")

    if right > left:
        selected_global_rows = selected_row_indices[left:right]
        local_rows = selected_global_rows - global_row_start

        # Take selected rows from this Arrow batch
        table = pa.Table.from_batches([record_batch])
        table = table.take(pa.array(local_rows.astype(np.int64)))
        table = table.cast(cast_schema, safe=False)

        pdf_batch = table.to_pandas(
            split_blocks=True,
            self_destruct=True
        )

        batch_arr = pdf_batch.to_numpy(
            dtype=np.float32,
            copy=True
        )

        # First column is target
        y_batch = batch_arr[:, 0]

        bad_y_mask = ~np.isfinite(y_batch)
        if bad_y_mask.any():
            y_batch[bad_y_mask] = 0.0

        # Remaining columns are model features
        X_batch_all = batch_arr[:, 1:]

        bad_x_mask = np.isinf(X_batch_all)
        if bad_x_mask.any():
            X_batch_all[bad_x_mask] = np.nan

        # Predict SHAP in smaller batches
        for pred_start in range(0, X_batch_all.shape[0], PRED_BATCH_SIZE):
            pred_end = min(pred_start + PRED_BATCH_SIZE, X_batch_all.shape[0])

            X_pred = X_batch_all[pred_start:pred_end, :]

            shap_pred_all = booster.predict(
                X_pred,
                pred_contrib=True,
                num_threads=NUM_THREADS
            )

            target_values = y_batch[pred_start:pred_end]

            # Extract selected feature values and selected SHAP values
            for feature_name, feature_idx in zip(FEATURE_NAME_LIST, FEATURE_INDEX_LIST):
                feature_values = X_pred[:, feature_idx]
                shap_values = shap_pred_all[:, feature_idx]

                dependence_store[feature_name]["feature_value"].append(
                    feature_values.astype(np.float32)
                )

                dependence_store[feature_name]["shap_value"].append(
                    shap_values.astype(np.float32)
                )

                dependence_store[feature_name]["target_value"].append(
                    target_values.astype(np.float32)
                )

            del X_pred, shap_pred_all
            gc.collect()

        selected_pointer += len(local_rows)

        elapsed = time.time() - start_time

        print(
            f"Batch {batch_id} | collected rows: {selected_pointer:,} / {len(selected_row_indices):,} "
            f"| elapsed sec: {elapsed:,.1f}"
        )

        del pdf_batch, batch_arr, X_batch_all, y_batch, table
        gc.collect()

    global_row_start = global_row_end

    if selected_pointer >= len(selected_row_indices):
        break

elapsed = time.time() - start_time

print("===================================================")
print("SHAP DEPENDENCE DATA COMPLETE")
print("===================================================")
print("Rows collected:", selected_pointer)
print(f"Elapsed seconds: {elapsed:,.2f}")
print("===================================================")

In [0]:
# ============================================================
# Create, save, and plot SHAP dependence for each feature
# ============================================================

all_dependence_dfs = []

for feature_name in FEATURE_NAME_LIST:
    safe_feature = safe_filename(feature_name)

    shap_dependence_df = pd.DataFrame({
        "feature": feature_name,
        "feature_value": np.concatenate(dependence_store[feature_name]["feature_value"]),
        "shap_value": np.concatenate(dependence_store[feature_name]["shap_value"]),
        "target_value": np.concatenate(dependence_store[feature_name]["target_value"])
    })

    shap_dependence_df = shap_dependence_df.replace([np.inf, -np.inf], np.nan)

    shap_dependence_df = (
        shap_dependence_df
        .dropna(subset=["feature_value", "shap_value"])
        .reset_index(drop=True)
    )

    all_dependence_dfs.append(shap_dependence_df)

    print("===================================================")
    print("Creating plot for:", feature_name)
    print("Rows:", shap_dependence_df.shape[0])
    print("===================================================")

    # Create binned trend line
    plot_df = shap_dependence_df.copy()

    try:
        plot_df["feature_bin"] = pd.qcut(
            plot_df["feature_value"].rank(method="first"),
            q=20,
            duplicates="drop"
        )

        binned_df = (
            plot_df
            .groupby("feature_bin", observed=True)
            .agg(
                avg_feature_value=("feature_value", "mean"),
                avg_shap_value=("shap_value", "mean"),
                median_shap_value=("shap_value", "median"),
                n_rows=("shap_value", "size")
            )
            .reset_index()
        )

    except Exception:
        binned_df = pd.DataFrame({
            "avg_feature_value": [plot_df["feature_value"].mean()],
            "avg_shap_value": [plot_df["shap_value"].mean()],
            "median_shap_value": [plot_df["shap_value"].median()],
            "n_rows": [len(plot_df)]
        })

    # Save raw data
    local_data_path = os.path.join(
        LOCAL_OUTPUT_DIR,
        f"shap_dependence_data_{safe_feature}.csv"
    )

    shap_dependence_df.to_csv(local_data_path, index=False)

    copy_local_to_output(
        local_data_path,
        f"shap_dependence_data_{safe_feature}.csv"
    )

    # Save binned data
    local_binned_path = os.path.join(
        LOCAL_OUTPUT_DIR,
        f"shap_dependence_binned_trend_{safe_feature}.csv"
    )

    binned_df.to_csv(local_binned_path, index=False)

    copy_local_to_output(
        local_binned_path,
        f"shap_dependence_binned_trend_{safe_feature}.csv"
    )

    # Plot
    plt.figure(figsize=(11, 7))

    plt.scatter(
        plot_df["feature_value"],
        plot_df["shap_value"],
        s=8,
        alpha=0.25
    )

    plt.plot(
        binned_df["avg_feature_value"],
        binned_df["avg_shap_value"],
        marker="o",
        linewidth=2,
        label="Binned average SHAP"
    )

    plt.axhline(0, linestyle="--", linewidth=1)

    plt.xlabel(feature_name)
    plt.ylabel("SHAP contribution to prediction")
    plt.title(f"SHAP Dependence Plot: {feature_name}")

    plt.grid(True)
    plt.legend()
    plt.tight_layout()

    local_plot_path = os.path.join(
        LOCAL_OUTPUT_DIR,
        f"shap_dependence_plot_{safe_feature}.png"
    )

    plt.savefig(
        local_plot_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    copy_local_to_output(
        local_plot_path,
        f"shap_dependence_plot_{safe_feature}.png"
    )

In [0]:
# ============================================================
# Optional: save combined long-format dependence data
# ============================================================

combined_dependence_df = pd.concat(
    all_dependence_dfs,
    ignore_index=True
)

local_combined_path = os.path.join(
    LOCAL_OUTPUT_DIR,
    "shap_dependence_data_all_requested_features.csv"
)

combined_dependence_df.to_csv(local_combined_path, index=False)

copy_local_to_output(
    local_combined_path,
    "shap_dependence_data_all_requested_features.csv"
)

print("Combined dependence data shape:")
print(combined_dependence_df.shape)

combined_dependence_df.head()